In [0]:
%run ./01-config

In [0]:
# Import core PySpark SQL functions (e.g., current_timestamp, from_json, window functions)
from pyspark.sql import functions as F
# Import Window for defining partitioned and ordered operations (e.g., ranking records)
from pyspark.sql.window import Window
from pyspark.sql.functions import from_unixtime

In [0]:
silver_output = dbutils.jobs.taskValues.get(taskKey = "Silver", key = "silver_output")

# Access the data
catalog = silver_output.get("catalog", "")
schema_2 = silver_output.get("schema_2", "")
schema_3 = silver_output.get("schema_3", "")



In [0]:
#   Construct absolute paths for raw data ingestion and streaming checkpoints
# - `landing_zone`: Base directory where raw source files (CSV/JSON) are staged for ingestion

base_dir_data = spark.sql("describe external location `data-zone`").select("url").collect()[0][0]
base_dir_checkpoint = spark.sql("describe external location `checkpoints-zone`").select("url").collect()[0][0]

In [0]:
# - `checkpoint_base`: Directory to store streaming query offsets and state (required for fault tolerance)

TRAIN_PERFORMANCE_KPI = base_dir_checkpoint + "/gold/train_performance_kpi"

In [0]:
# # Switch to the catalog
# spark.sql("USE CATALOG dev")

# Create the schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:

# Creates a Gold `train_performance_kpi` table with cleaned gold_train_performance_kpi.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_3}.gold_train_performance_kpi (
    full_date DATE,
    year INT,
    month INT,
    month_name STRING,
    day_name STRING,
    hour INT,
    operating_company STRING,
    current_station STRING,
    next_station STRING,
    total_movements BIGINT,
    avg_delay_minutes DOUBLE,
    time_variance DOUBLE,
    on_time_percentage DOUBLE,
    severe_delay_rate DOUBLE,
    station_compliance_rate DOUBLE,
    extreme_temp_avg_delay DOUBLE,
    peak_hour_avg_delay DOUBLE,
    train_reliability_index DOUBLE,
    congestion_index DOUBLE
)
USING DELTA
""")

In [0]:
def upserter(df_micro_batch, batch_id, merge_query, temp_view):
    """
    Generic helper to execute a Delta MERGE operation on a micro-batch DataFrame.

    - Creates a temporary view from the micro-batch so it can be referenced in SQL MERGE
    - Executes the provided `merge_query`
    - Logs batch completion for observability

    Intended for reuse across Silver tables via foreachBatch.
    """
    df_micro_batch.createOrReplaceTempView(temp_view)
    df_micro_batch._jdf.sparkSession().sql(merge_query)
    print(f"Batch {batch_id} for {temp_view} processed.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

def upsert_gold_train_performance(catalog: str, schema_2: str, schema_3: str, once=True, processing_time="15 seconds"):
    """
    Creates/updates the Gold Train Performance table from the Silver fact and dimension tables.

    - Joins fact_train_performance with dim_date and dim_operators
    - Calculates KPIs like avg_delay, on_time_percentage, train_reliability_index, congestion_index
    - Uses foreachBatch + MERGE into Gold Delta table
    """

    # -------------------------
    # 1️⃣ Define MERGE Query
    # -------------------------
    merge_query = f"""
    MERGE INTO {catalog}.{schema_3}.gold_train_performance_kpi a
    USING gold_train_performance_cdc b
    ON a.full_date = b.full_date
       AND a.hour = b.hour
       AND a.operating_company = b.operating_company
       AND a.current_station = b.current_station
       AND a.next_station = b.next_station

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """

    # -------------------------
    # 2️⃣ Read Streaming Source
    # -------------------------
    df_stream = (
        spark.readStream
            .table(f"{catalog}.{schema_2}.fact_train_performance")
            .alias("f")
            .join(
                spark.table(f"{catalog}.{schema_2}.dim_date").alias("d"),
                F.col("f.date_key") == F.col("d.date_key")
            )
            .join(
                spark.table(f"{catalog}.{schema_2}.dim_operators").alias("o"),
                F.col("f.toc_id") == F.col("o.toc_id")
            )
             # ✅ FILTER NULL TEMPERATURE HERE
            .filter(F.col("f.temperature").isNotNull())
            .withColumn("hour", F.hour("actual_timestamp"))
            .withColumn(
                "delay_minutes",
                F.round((F.unix_timestamp("actual_timestamp") - F.unix_timestamp("planned_timestamp")) / 60, 2)
            )
            .filter(F.col("temperature").isNotNull())
    )

    # -------------------------
    # 3️⃣ Aggregate Metrics per Batch
    # -------------------------
    def prepare_batch(batch_df: DataFrame, batch_id: int):
        agg_df = (
            batch_df.groupBy(
                "full_date",
                "year",
                "month",
                "month_name",
                "day_name",
                "hour",
                "operating_company",
                "current_station",
                "next_station"
            )
            .agg(
                F.count("train_id").alias("total_movements"),
                F.avg("delay_minutes").alias("avg_delay_minutes"),
                F.stddev("delay_minutes").alias("time_variance"),
                (F.sum(F.when(F.col("delay_minutes") <= 5, 1).otherwise(0)) / F.count("*")).alias("on_time_percentage"),
                (F.sum(F.when(F.col("delay_minutes") > 15, 1).otherwise(0)) / F.count("*")).alias("severe_delay_rate"),
                (F.sum(F.when(F.col("delay_minutes") <= 5, 1).otherwise(0)) / F.count("*")).alias("station_compliance_rate"),
                F.avg(F.when((F.col("temperature") < 0) | (F.col("temperature") > 30), F.col("delay_minutes"))).alias("extreme_temp_avg_delay"),
                F.avg(F.when((F.col("hour").between(7,9)) | (F.col("hour").between(16,18)), F.col("delay_minutes"))).alias("peak_hour_avg_delay")
            )
            .withColumn(
                "train_reliability_index",
                F.col("on_time_percentage") * 0.7 - F.col("severe_delay_rate") * 0.3
            )
            .withColumn(
                "congestion_index",
                F.col("total_movements") * F.col("avg_delay_minutes")
            )
        )

        # Use the existing upserter function to MERGE into Gold table
        upserter(agg_df, batch_id, merge_query, temp_view="gold_train_performance_cdc")

    # -------------------------
    # 4️⃣ Start Streaming Write
    # -------------------------
    stream_writer = (
        df_stream.writeStream
            .foreachBatch(prepare_batch)
            .outputMode("append")  # Append is correct; upserts handled inside foreachBatch
            .option("checkpointLocation", TRAIN_PERFORMANCE_KPI)
            .queryName("gold_train_performance_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()

In [0]:
upsert_gold_train_performance(catalog = catalog, schema_2 = schema_2, schema_3 = schema_3)

In [0]:
# %sql
# select * from dev.gold.gold_train_performance_kpi
# limit 5

In [0]:
def assert_zero(catalog, schema_3, table_name, condition):
    """
    Assert that no records match a bad condition.
    """
    print(f"Checking {table_name} for condition: {condition}")

    count = (
        spark.read.table(f"{catalog}.{schema_3}.{table_name}")
        .where(condition)
        .count()
    )

    assert count == 0, f"{count} invalid records found in {table_name} where {condition}"

    print("✔ Passed")


def assert_not_empty(catalog, schema_3, table_name):
    """
    Ensure table is not empty.
    """
    print(f"Checking {table_name} is not empty")

    count = spark.read.table(f"{catalog}.{schema_3}.{table_name}").count()

    assert count > 0, f"{table_name} is empty"

    print(f"✔ {count} records found")

In [0]:

print("\n🔎 Validating Gold Layer")


# Not Empty
assert_not_empty(catalog, schema_3, "gold_train_performance_kpi")


print("✔ Gold validation completed")